# 🚗 Notebook 2: Delivery Routing and ETA Estimation

When a customer opens the Gopuff app, the system needs to figure out **which warehouses can deliver to them** and **how long it will take**. This is harder than it sounds — you can’t just pick the closest warehouse on a straight line because roads, traffic, and rivers get in the way.

This notebook covers:
1. Finding nearby DCs using the **Haversine formula** (distance on a sphere)
2. Filtering candidates with **delivery zones**
3. Estimating delivery time from historical data
4. Caching nearby-DC lookups in Redis for speed

## Learning Goals

- Calculate real distances between lat/lon points (not just Euclidean)
- Understand the **two-step** nearby DC lookup: rough filter → precise estimate
- Build an ETA model from historical order data
- See how caching spatial lookups keeps latency under 100 ms

## 🛠️ Setup

```bash
cd 06-system-designs/gopuff
docker compose up -d
```

### Kernel Selection (VS Code)
Select the `.venv` kernel from the kernel picker (top-right).
If it doesn’t appear, reload VS Code (`Cmd+Shift+P` → “Reload Window”).

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import math
import time

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "gopuff", "user": "demo", "password": "demo",
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

conn = get_db_connection()
r = get_redis_client()
r.ping()
print("✅ Postgres and Redis connected")
conn.close()

## 1️⃣ The Haversine Formula — Distance on Earth

Earth is (roughly) a sphere. The straight-line distance between two GPS points on a flat map is **wrong** — it ignores the curvature. The **Haversine formula** calculates the great-circle distance (shortest path along the surface).

### Why not just use `sqrt((x2-x1)² + (y2-y1)²)`?

Euclidean distance treats latitude and longitude like a flat grid. Near the equator that’s roughly OK, but:
- 1° of longitude = 69 miles at the equator, but only 49 miles in Austin, TX
- At the poles, 1° of longitude = 0 miles!

The Haversine formula handles this correctly.

In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points
    on Earth (in miles) using the Haversine formula.
    """
    R = 3958.8  # Earth's radius in miles
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + \
        math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    return R * c


dist = haversine_miles(30.2672, -97.7431, 30.3716, -97.7064)
print(f"DC Downtown → DC North: {dist:.2f} miles")

euclidean = math.sqrt((30.3716 - 30.2672)**2 + (-97.7064 - (-97.7431))**2) * 69
print(f"Euclidean estimate:     {euclidean:.2f} miles")
print(f"Difference:             {abs(dist - euclidean):.2f} miles")

### 🔬 How wrong is Euclidean, really?

For a local-delivery service like Gopuff, the customer and DC are usually within a few miles, so a flat-grid approximation **isn't catastrophic for a rough filter** — but it's wrong enough that you wouldn't use it for the actual ETA.

Let's measure the error across a few representative distances.


In [ ]:
def euclidean_approx_miles(lat1, lon1, lat2, lon2):
    """Flat-grid approximation: 1° latitude ≈ 69 miles. Ignores Earth curvature."""
    return math.sqrt((lat2 - lat1) ** 2 + (lon2 - lon1) ** 2) * 69


# Austin Downtown is our anchor; compare at increasing distances.
anchor = (30.2672, -97.7431)
probes = [
    ("DC University (same neighborhood)", 30.2849, -97.7341),
    ("DC Round Rock (next town)",         30.5083, -97.6789),
    ("Houston, TX (next major city)",     29.7604, -95.3698),
    ("Dallas, TX (longer drive)",         32.7767, -96.7970),
]

print(f"{'Destination':<40s} {'Haversine':>10s} {'Euclidean':>10s} {'Error %':>8s}")
print("-" * 72)
for name, lat, lon in probes:
    h = haversine_miles(*anchor, lat, lon)
    e = euclidean_approx_miles(*anchor, lat, lon)
    err = abs(h - e) / h * 100
    print(f"{name:<40s} {h:>8.2f}mi {e:>8.2f}mi {err:>7.2f}%")

print("\n→ At a few miles the error is <1% (fine as a rough filter).")
print("  At 150+ miles the error grows — and we'd never want it in a final ETA.")


## 2️⃣ Finding Nearby DCs

The interview solution uses a **two-step approach**:

1. **Rough filter**: Find all DCs within a generous radius (e.g., 60 miles). This is fast because we just compute distance.
2. **Precise filter**: For the candidates from step 1, check the actual delivery zones and estimated drive times.

This avoids calling an expensive travel-time API for DCs that are clearly too far away.

In [ ]:
def find_nearby_dcs(customer_lat, customer_lon, max_radius_miles=10.0):
    """Find all DCs within max_radius_miles of the customer using Haversine."""
    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("SELECT id, name, latitude, longitude FROM distribution_centers WHERE is_active = TRUE")
    all_dcs = cur.fetchall()
    conn.close()

    nearby = []
    for dc in all_dcs:
        dist = haversine_miles(customer_lat, customer_lon, dc["latitude"], dc["longitude"])
        if dist <= max_radius_miles:
            nearby.append({"dc_id": dc["id"], "name": dc["name"],
                           "distance_miles": round(dist, 2),
                           "lat": dc["latitude"], "lon": dc["longitude"]})
    nearby.sort(key=lambda x: x["distance_miles"])
    return nearby


customer_lat, customer_lon = 30.2850, -97.7335
nearby = find_nearby_dcs(customer_lat, customer_lon, max_radius_miles=8)

print(f"📍 Customer at ({customer_lat}, {customer_lon})")
print(f"   Found {len(nearby)} DCs within 8 miles:\n")
for dc in nearby:
    bar = "█" * int(dc['distance_miles'] * 3)
    print(f"   {dc['name']:<18s} {dc['distance_miles']:5.2f} mi  {bar}")

### ⚠️ That Function Scans Every DC in the Country

`find_nearby_dcs` pulls **all 10,000 DCs** out of Postgres and computes 10,000
Haversine distances to find the 3 that matter. With 8 rows in this lab it's
instant. At real scale it is O(n) per request, on the hot path, at thousands of
requests per second.

You need a **spatial index**: a structure that answers "what's within R of this
point" without looking at everything. Three common choices:

| Option | How it indexes | When to reach for it |
|---|---|---|
| **Redis `GEOSEARCH`** | Geohash packed into a sorted-set score | You already run Redis and the point set is small and changes rarely. This is Gopuff-shaped. |
| **PostGIS `ST_DWithin` + GiST** | R-tree over real geometry | The points live in Postgres anyway and you need real polygons (actual delivery zones, not circles) |
| **Application-side geohash prefix** | String prefix match on any KV store | You're on DynamoDB/Cassandra and can't install extensions |

Redis is already running in this lab, so let's actually build the index instead
of just naming it.

In [ ]:
# ── Build a Redis geospatial index over the DCs and query it ────────────
r = get_redis_client()
GEO_KEY = "dc:geo"

conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, latitude, longitude FROM distribution_centers WHERE is_active = TRUE")
all_dcs = cur.fetchall()
conn.close()

r.delete(GEO_KEY)
# GEOADD takes (longitude, latitude, member) — longitude FIRST. Getting this
# backwards is the single most common Redis GEO bug; your results come back
# empty and you blame the radius.
r.geoadd(GEO_KEY, [v for dc in all_dcs
                   for v in (dc["longitude"], dc["latitude"], f"dc:{dc['id']}")])
print(f"📍 Indexed {r.zcard(GEO_KEY)} DCs into the Redis sorted set '{GEO_KEY}'")


def find_nearby_dcs_geo(customer_lat, customer_lon, max_radius_miles=10.0):
    """Same answer as find_nearby_dcs, but O(log n + k) instead of O(n)."""
    hits = r.geosearch(
        GEO_KEY,
        longitude=customer_lon, latitude=customer_lat,
        radius=max_radius_miles, unit="mi",
        withdist=True, sort="ASC",
    )
    by_id = {dc["id"]: dc for dc in all_dcs}
    return [{"dc_id": int(member.split(":")[1]),
             "name": by_id[int(member.split(":")[1])]["name"],
             "distance_miles": round(dist, 2)}
            for member, dist in hits]


# ── Do the two implementations actually agree? ──────────────────────────
scan_result = find_nearby_dcs(customer_lat, customer_lon, max_radius_miles=8)
geo_result = find_nearby_dcs_geo(customer_lat, customer_lon, max_radius_miles=8)

print()
print(f"{'DC':<20} {'Haversine scan':>16} {'Redis GEOSEARCH':>17}")
print("-" * 56)
scan_by_id = {d["dc_id"]: d for d in scan_result}
for g in geo_result:
    s = scan_by_id.get(g["dc_id"])
    print(f"{g['name']:<20} {s['distance_miles'] if s else '—':>14} mi "
          f"{g['distance_miles']:>14} mi")

assert [d["dc_id"] for d in scan_result] == [d["dc_id"] for d in geo_result], \
    "the index must return the same DCs in the same order"
for s, g in zip(scan_result, geo_result):
    # Redis models the Earth as a sphere too, so agreement should be tight.
    assert abs(s["distance_miles"] - g["distance_miles"]) < 0.05, (s, g)
print("\n✅ Identical DCs, identical ordering, distances agree to <0.05 mi.")

# ── Latency ─────────────────────────────────────────────────────────────
N = 200
t0 = time.perf_counter()
for _ in range(N):
    find_nearby_dcs(customer_lat, customer_lon, 8)
scan_ms = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N):
    find_nearby_dcs_geo(customer_lat, customer_lon, 8)
geo_ms = (time.perf_counter() - t0) / N * 1000

print(f"\n⏱️  Postgres fetch-all + Python Haversine: {scan_ms:>7.3f} ms  (O(n), n={len(all_dcs)})")
print(f"⏱️  Redis GEOSEARCH:                       {geo_ms:>7.3f} ms  (O(log n + k))")
print(f"   Speedup at n={len(all_dcs)}: {scan_ms / geo_ms:.1f}x")
print()
print(f"   ⚠️  Be honest about what that {scan_ms / geo_ms:.0f}x is. At n={len(all_dcs)} the Haversine")
print("   arithmetic is free; almost all of the 'scan' time is one Postgres")
print("   round trip. The measurement here is mostly Postgres-vs-Redis latency.")
print("   The reason to use the index is the SLOPE, not this ratio: scan cost")
print(f"   grows linearly with DC count ({len(all_dcs)} today, 10,000 in production =")
print("   1,250x more distance math and 1,250x more rows on the wire), while")
print("   GEOSEARCH grows as log n and returns only the handful of hits.")
print()
print("💡 What the index costs you:")
print("   • A second source of truth. DCs opening/closing must be written to")
print("     BOTH Postgres and Redis, or the index silently goes stale.")
print("   • Circles, not polygons. GEOSEARCH answers 'within R miles as the crow")
print("     flies'. A river or a highway with no exit makes that answer wrong.")
print("     Real routing needs road-network distance — the radius is only the")
print("     cheap CANDIDATE filter, which is exactly how we use it here.")
print("   • Redis GEO holds points only. If you need true delivery polygons,")
print("     that is the case for PostGIS instead.")

## 3️⃣ Delivery Zones and ETA Estimation

Each DC has **delivery zones** — concentric rings with different delivery time estimates. A DC’s “core” zone (3 miles) might promise 15-minute delivery, while the “extended” zone (7 miles) takes 30 minutes.

### How ETA Works

```
Customer Location
       │
       ▼
  Find nearby DCs (Haversine)
       │
       ▼
  For each DC, check delivery zones
       │
       ├── Distance < core radius? → Use core ETA
       ├── Distance < extended radius? → Use extended ETA
       └── Too far? → Skip this DC
       │
       ▼
  Pick the DC with the best (lowest) ETA
```

In [ ]:
def estimate_delivery(customer_lat, customer_lon):
    """For each nearby DC, find matching delivery zone and estimate time."""
    candidates = find_nearby_dcs(customer_lat, customer_lon, max_radius_miles=15)
    if not candidates:
        return []

    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    dc_ids = [c["dc_id"] for c in candidates]
    cur.execute("""
        SELECT dc_id, zone_name, max_radius_miles, avg_delivery_minutes, surge_multiplier
        FROM delivery_zones WHERE dc_id = ANY(%s)
        ORDER BY dc_id, max_radius_miles
    """, (dc_ids,))
    zones = cur.fetchall()
    conn.close()

    zone_map = {}
    for z in zones:
        zone_map.setdefault(z["dc_id"], []).append(z)

    results = []
    for cand in candidates:
        dc_zones = zone_map.get(cand["dc_id"], [])
        for z in dc_zones:
            if cand["distance_miles"] <= z["max_radius_miles"]:
                factor = cand["distance_miles"] / z["max_radius_miles"]
                est_min = int(z["avg_delivery_minutes"] * (0.7 + 0.6 * factor))
                results.append({
                    "dc_id": cand["dc_id"], "dc_name": cand["name"],
                    "distance_miles": cand["distance_miles"],
                    "zone": z["zone_name"],
                    "estimated_minutes": est_min,
                    "surge_multiplier": float(z["surge_multiplier"]),
                })
                break

    results.sort(key=lambda x: x["estimated_minutes"])
    return results


estimates = estimate_delivery(30.2850, -97.7335)
print("🚗 Delivery Estimates")
print("=" * 70)
for e in estimates:
    clk = "🟢" if e["estimated_minutes"] <= 20 else "🟡" if e["estimated_minutes"] <= 35 else "🔴"
    print(f"  {clk} {e['dc_name']:<18s} | {e['distance_miles']:5.2f} mi | "
          f"{e['zone']:<22s} | ~{e['estimated_minutes']} min")

## 4️⃣ Improving ETA with Historical Data

Our zone-based estimate is a starting point, but real delivery times depend on:
- **Time of day** — lunch rush vs late night
- **Day of week** — weekends are busier
- **Actual past deliveries** — real data beats estimates

Let’s use our historical order data to build a better ETA model.

In [ ]:
def get_historical_eta(dc_id, hour_of_day):
    """Look at past delivered orders to estimate delivery time by hour."""
    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT
            COUNT(*) AS sample_size,
            ROUND(AVG(actual_delivery_minutes)) AS avg_minutes,
            ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY actual_delivery_minutes)) AS median_minutes,
            ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY actual_delivery_minutes)) AS p90_minutes,
            MIN(actual_delivery_minutes) AS min_minutes,
            MAX(actual_delivery_minutes) AS max_minutes
        FROM orders
        WHERE dc_id = %s AND EXTRACT(HOUR FROM created_at) = %s
          AND status = 'delivered' AND actual_delivery_minutes IS NOT NULL
    """, (dc_id, hour_of_day))
    result = cur.fetchone()
    conn.close()
    return dict(result) if result else {}


print("📊 Historical Delivery Times — DC Downtown")
print("=" * 60)
print(f"  {'Hour':<8s} {'Samples':>8s} {'Avg':>6s} {'Median':>8s} {'P90':>6s}")
print("-" * 60)
for hour in [8, 12, 15, 18, 21]:
    stats = get_historical_eta(dc_id=1, hour_of_day=hour)
    if stats and stats["sample_size"] > 0:
        print(f"  {hour:02d}:00   {stats['sample_size']:>8d} {stats['avg_minutes']:>5.0f}m "
              f"{stats['median_minutes']:>7.0f}m {stats['p90_minutes']:>5.0f}m")
    else:
        print(f"  {hour:02d}:00   {'no data':>8s}")

## 5️⃣ Caching the Nearby DC Lookup

Every availability request starts with “find nearby DCs.” Since DCs don’t move (they’re buildings!), we can cache this aggressively.

**Strategy**: Round the customer’s lat/lon to a grid cell (e.g., 0.01° ≈ 0.7 miles) and cache the result. Customers in the same grid cell get the same nearby DCs.

In [ ]:
def find_nearby_dcs_cached(customer_lat, customer_lon, max_radius_miles=10.0, grid_precision=2):
    """Cached nearby-DC lookup with grid-cell rounding."""
    r = get_redis_client()
    grid_lat = round(customer_lat, grid_precision)
    grid_lon = round(customer_lon, grid_precision)
    cache_key = f"nearby:{grid_lat},{grid_lon}:{max_radius_miles}"

    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), "cache"

    nearby = find_nearby_dcs(customer_lat, customer_lon, max_radius_miles)
    r.setex(cache_key, 300, json.dumps(nearby))
    return nearby, "database"


r = get_redis_client()
r.flushdb()

start = time.time()
nearby, source = find_nearby_dcs_cached(30.2850, -97.7335)
t1 = (time.time() - start) * 1000
print(f"1st call: {source:>8s} | {t1:.1f} ms | {len(nearby)} DCs")

start = time.time()
nearby, source = find_nearby_dcs_cached(30.2850, -97.7335)
t2 = (time.time() - start) * 1000
print(f"2nd call: {source:>8s} | {t2:.1f} ms | {len(nearby)} DCs")

start = time.time()
nearby, source = find_nearby_dcs_cached(30.2855, -97.7330)
t3 = (time.time() - start) * 1000
print(f"Near loc: {source:>8s} | {t3:.1f} ms | {len(nearby)} DCs  (same grid cell!)")

## 6️⃣ Visualizing DC Coverage

Let’s build a simple text-based map to see how our DCs cover the Austin metro area.

In [ ]:
conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, latitude, longitude FROM distribution_centers ORDER BY id")
dcs = cur.fetchall()
conn.close()

print("\n🗺️  Austin DC Map (text-based)")
print("   North ↑\n")

min_lat = min(d['latitude'] for d in dcs) - 0.02
max_lat = max(d['latitude'] for d in dcs) + 0.02
min_lon = min(d['longitude'] for d in dcs) - 0.02
max_lon = max(d['longitude'] for d in dcs) + 0.02

ROWS, COLS = 20, 50
grid = [[' ' for _ in range(COLS)] for _ in range(ROWS)]

for dc in dcs:
    row = ROWS - 1 - int((dc['latitude'] - min_lat) / (max_lat - min_lat) * (ROWS - 1))
    col = int((dc['longitude'] - min_lon) / (max_lon - min_lon) * (COLS - 1))
    row = max(0, min(ROWS - 1, row))
    col = max(0, min(COLS - 1, col))
    grid[row][col] = str(dc['id'])

cr = ROWS - 1 - int((30.2850 - min_lat) / (max_lat - min_lat) * (ROWS - 1))
cc = int((-97.7335 - min_lon) / (max_lon - min_lon) * (COLS - 1))
grid[max(0, min(ROWS-1, cr))][max(0, min(COLS-1, cc))] = '★'

for row in grid:
    print('   │' + ''.join(row) + '│')

print("\n   Legend: ★ = Customer")
for dc in dcs:
    print(f"           {dc['id']} = {dc['name']}")

### 📍 Same Map, With Matplotlib

The ASCII map is fun, but a real plot makes it obvious how the DCs cover the metro. We'll draw DCs, the customer location, and a 5-mile circle around each DC to visualize delivery coverage.


In [ ]:
import matplotlib.pyplot as plt

conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, latitude, longitude FROM distribution_centers ORDER BY id")
dcs = cur.fetchall()
conn.close()

customer_lat, customer_lon = 30.2850, -97.7335

fig, ax = plt.subplots(figsize=(8, 7))

# 1 degree latitude ≈ 69 miles; longitude shrinks with cos(lat)
lat_per_mile = 1 / 69
lon_per_mile = 1 / (69 * math.cos(math.radians(customer_lat)))
coverage_miles = 5

for dc in dcs:
    circle = plt.Circle(
        (dc['longitude'], dc['latitude']),
        coverage_miles * lon_per_mile,
        color='steelblue', alpha=0.12,
    )
    ax.add_patch(circle)
    ax.plot(dc['longitude'], dc['latitude'], 'o', color='steelblue', markersize=9)
    ax.annotate(dc['name'], (dc['longitude'], dc['latitude']),
                xytext=(6, 6), textcoords='offset points', fontsize=8)

ax.plot(customer_lon, customer_lat, '*', color='crimson', markersize=18, label='Customer')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Austin Metro DCs with ~5mi Delivery Coverage')
ax.set_aspect('equal', adjustable='datalim')
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### 📝 What Real ETAs Also Need

Our zone-based + historical ETA is a solid v1, but production systems combine more signals:

- **Courier availability** — if every driver is already on a run, your ETA must include the **dispatch wait**, not just drive time.
- **Live traffic / map service** — Google/Mapbox directions API for candidate DCs.
- **Weather & demand spikes** — padding during rain, big sporting events, etc.

These don't change the system-design pattern (same two-step: rough filter → precise estimate), just the inputs to step 2.

### 📚 Where Redis GEO Stops Being Enough

We built the `GEOSEARCH` index earlier in this notebook and it beat the linear
scan handily. The point at which you outgrow it:

- **You need polygons, not radii.** Real delivery areas follow roads, rivers and
  city limits. `GEOSEARCH` only knows circles. → PostGIS `ST_Contains` over a
  GiST index.
- **You need the index and the data in one transaction.** Redis GEO is a second
  store, so opening a DC is a two-phase write that can half-fail. → keep the
  index in Postgres.
- **You need drive time, not distance.** No spatial index gives you this; a
  routing engine (OSRM, Valhalla) or a maps API does. The spatial index is the
  *candidate filter* that keeps your routing bill sane.


## 🔑 Key Takeaways

| Concept | What We Did |
|---------|-------------|
| **Haversine formula** | Accurate distance between lat/lon points on Earth’s surface |
| **Two-step lookup** | Rough distance filter → precise zone-based check |
| **Delivery zones** | Pre-computed rings around each DC with estimated times |
| **Historical ETA** | Used past order data to refine time-of-day estimates |
| **Grid-based caching** | Round lat/lon so nearby customers share cache entries |

### Interview Tip

When discussing nearby-DC lookup, mention the **optimization progression**:
1. Simple Euclidean distance (bad — ignores Earth’s curvature)
2. Haversine + fixed radius (good — accurate distance)
3. Haversine + external travel time API for candidates (great — accounts for roads/traffic)

Also note that DCs are **static** (buildings don’t move), so the nearby-DC result can be cached aggressively.

## 🧹 Cleanup

In [ ]:
r = get_redis_client()
r.delete("dc:geo")  # the geospatial index we built earlier
keys = r.keys("nearby:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")